In [52]:
import numpy as np
import pandas as pd
from sklearn.model_selection import TimeSeriesSplit, cross_validate
from keras.utils import set_random_seed
from rnn_models import build_model

In [53]:
LAG = 12

## Read in and window data

In [54]:
flights = pd.read_csv("./Pre-processed datasets/flights_final.csv", index_col=0, parse_dates=True)

In [58]:
original_passengers = pd.read_csv("Datasets/airline-passengers.csv", index_col=0, parse_dates=True)["Passengers"]
log_passengers = np.log(original_passengers)
target_dates = flights.index[LAG:]          # the month each y belongs to

def to_passengers(z_pred, dates):
    pos = log_passengers.index.get_indexer(dates)
    log_pred = z_pred + log_passengers.iloc[pos-1].values + log_passengers.iloc[pos-12].values - log_passengers.iloc[pos-13].values
    return np.exp(log_pred)

In [59]:
# Turn the series into windows.
series = flights["Passengers"].to_numpy(dtype="float32")
X = np.stack([series[i:i + LAG] for i in range(len(s) - LAG)])
y = series[LAG:]

## Cross-validation function

In [64]:
def cv_scores(architecture, units=8, learning_rate=0.01, epochs=300, n_splits=5, test_size=12, seed=0, **kw):
    cv = TimeSeriesSplit(n_splits=n_splits, test_size=test_size)
    scores = []
    for train_index, test_index in cv.split(X):
        set_random_seed(seed)

        # scale using the training fold only.
        mean, sd = y[train_index].mean(), y[train_index].std()
        X_train, y_train = (X[train_index] - mean)/sd, (y[train_index] - mean)/sd
        X_test = (X[test_index] - mean)/sd

        # Build and fit model.
        model = build_model(architecture, lags=X.shape[1], units=units, lr=learning_rate, **kw)
        model.fit(X_train, y_train, epochs=epochs, batch_size=len(y_train), verbose=0)

        # Test the model.
        pred = model.predict(X_test, verbose=0).ravel()*sd + mean

        # Reverse pre-preocessing steps before calculating errors.
        dates = target_dates[test_index]
        pred_pass = to_passengers(pred, dates)
        actual = original_passengers.loc[dates].values
        scores.append(np.sqrt(np.mean((pred_pass - actual) ** 2)))   # RMSE in passengers
        
    return scores

## Elman: cross validation

In [65]:
scores = pd.DataFrame({
    "elman": cv_scores("elman"),
    "jordan": cv_scores("jordan"),
    "mrnn": cv_scores("mrnn", n_banks=4),
}, index=[f"fold {i+1}" for i in range(5)])

scores.loc["mean"] = scores.mean()

In [66]:
scores

,elman,jordan,mrnn
fold 1,13.809450,7.377849,15.066929
fold 2,17.332644,5.813897,22.825410
fold 3,13.096547,13.566575,20.594715
fold 4,21.321606,20.688345,23.275229
fold 5,25.719295,24.524018,25.782405
mean,18.255908,14.394137,21.508938
